# Experiment 57 - Taxon Prior TTA Plateau

**Building on:** exp46 (OOF cmAP=0.9558), the stronger taxon-prior OOF branch.

**Hypothesis:** exp49-53 showed that non-uniform TTA weighting and more aggressive prior scaling improve OOF in the exp22 family. Apply that idea to the taxon-aware exp46 branch: re-search event/texture lambdas, 0s/2.5s TTA weight, power, alpha_texture, and nearby ensemble weights.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp46_path = nb_dir / 'exp46_no_proto.ipynb'
exp46_nb = json.loads(exp46_path.read_text())

def exp46_source(cell_idx):
    src = ''.join(exp46_nb['cells'][cell_idx]['source'])
    if cell_idx == 1:
        src = src.replace("PROJECT_ROOT = Path('/Users/jjannik/Development/kaggle/birdclef')",
                          "PROJECT_ROOT = Path.cwd()\nif PROJECT_ROOT.name == 'notebooks':\n    PROJECT_ROOT = PROJECT_ROOT.parent")
    return src

# Reuse exp46 setup through OOF component generation; skip exp46's final grid cell.
for cell_idx in range(1, 10):
    print(f'--- executing exp46 cell {cell_idx} ---')
    src = exp46_source(cell_idx)
    exec(compile(src, f'exp46_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp57: taxon-aware prior + TTA plateau grid

LAM_EVENTS = [2.0, 2.8, 3.5, 5.0]
LAM_TEXTURES = [10.0, 15.0, 20.0, 30.0, 45.0]
TTA_W0S = [0.50, 0.70, 0.85, 0.90]
POWERS = [0.40, 0.70, 1.00]
ALPHA_TEXTURES = [0.20, 0.30]
ENSEMBLE_GRIDS = [
    (0.02, 0.60, 0.38),  # exp46 best
    (0.02, 0.58, 0.40),
    (0.00, 0.62, 0.38),
    (0.05, 0.57, 0.38),
    (0.02, 0.63, 0.35),
]

best_cmap = -1.0
best_cfg = {}
results = []
prior_cache = {}

def get_prior_pair_exp57(lam_event, lam_texture):
    key = (lam_event, lam_texture)
    if key not in prior_cache:
        prior_cache[key] = (
            apply_prior_taxon(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables,
                              lambda_event=lam_event, lambda_texture=lam_texture),
            apply_prior_taxon(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables,
                              lambda_event=lam_event, lambda_texture=lam_texture),
        )
    return prior_cache[key]

for lam_event in LAM_EVENTS:
    for lam_texture in LAM_TEXTURES:
        prior_0, prior_250 = get_prior_pair_exp57(lam_event, lam_texture)
        for wp, wm, wpr in ENSEMBLE_GRIDS:
            fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
            fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
            for tta_w0 in TTA_W0S:
                fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
                for power in POWERS:
                    for alpha_texture in ALPHA_TEXTURES:
                        probs = postprocess_taxon(fp_avg, power=power, alpha_event=0.10,
                                                  alpha_texture=alpha_texture)
                        cmap = padded_cmap(Y_FULL_aligned, probs)
                        auc = macro_auc(Y_FULL_aligned, probs)
                        row = {'lam_event': lam_event, 'lam_texture': lam_texture, 'tta_w0': tta_w0,
                               'power': power, 'alpha_texture': alpha_texture,
                               'wp': wp, 'wm': wm, 'wpr': wpr, 'auc': auc, 'cmap': cmap}
                        results.append(row)
                        if cmap > best_cmap:
                            best_cmap = cmap
                            best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 25 configs:')
print(df.head(25).to_string(index=False))
print('\nBest cmAP by lambda_event:')
print(df.groupby('lam_event')['cmap'].max().reset_index().to_string(index=False))
print('\nBest cmAP by lambda_texture:')
print(df.groupby('lam_texture')['cmap'].max().reset_index().to_string(index=False))
print('\nBest cmAP by tta_w0:')
print(df.groupby('tta_w0')['cmap'].max().reset_index().to_string(index=False))

prior_0_f, prior_250_f = get_prior_pair_exp57(best_cfg['lam_event'], best_cfg['lam_texture'])
fp_0_f = best_cfg['wp'] * oof_proto_0 + best_cfg['wm'] * oof_mlp_0 + best_cfg['wpr'] * prior_0_f
fp_250_f = best_cfg['wp'] * oof_proto_250 + best_cfg['wm'] * oof_mlp_250 + best_cfg['wpr'] * prior_250_f
fp_f = best_cfg['tta_w0'] * fp_0_f + (1.0 - best_cfg['tta_w0']) * fp_250_f
p_best = postprocess_taxon(fp_f, power=best_cfg['power'], alpha_event=0.10,
                           alpha_texture=best_cfg['alpha_texture'])
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)
cmap_tex = padded_cmap(Y_FULL_aligned[:, TEXTURE_MASK], p_best[:, TEXTURE_MASK])
cmap_evt = padded_cmap(Y_FULL_aligned[:, ~TEXTURE_MASK], p_best[:, ~TEXTURE_MASK])

print('=' * 60)
print('Experiment 57 - Taxon prior TTA plateau')
print(f"Best: lambda_event={best_cfg['lam_event']}, lambda_texture={best_cfg['lam_texture']}, tta_w0={best_cfg['tta_w0']}, power={best_cfg['power']}, alpha_texture={best_cfg['alpha_texture']}")
print(f"Weights: wp={best_cfg['wp']}, wm={best_cfg['wm']}, wpr={best_cfg['wpr']}")
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'Texture cmAP={cmap_tex:.5f}  Event cmAP={cmap_evt:.5f}')
print(f'vs exp46 OOF (0.95579): delta cmAP = {cmap_b - 0.95579:+.5f}')
print(f'vs exp53 OOF (0.94973): delta cmAP = {cmap_b - 0.94973:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
